In [1]:
from scripts.conf_file_finding import try_find_conf_file
try_find_conf_file()

Local configuration file found !!, no need to run the configuration (unless configuration has changed)


In [2]:
import datajoint as dj
import pandas as pd
import u19_pipeline.ephys_pipeline as ep

/mnt/cup/braininit/Shared/repos/TestU19PipelinePython2/U19-pipeline-python/.venv/lib/python3.13/site-packages/datajoint/plugin.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources  # requires setuptools<82
[2026-08-05 11:16:00,954][INFO]: DataJoint 0.14.9 connected to alvaros@datajoint00.pni.princeton.edu:3306


### Read all parameters for preprocessing

In [3]:
all_preparams = pd.DataFrame(ep.ephys_element.PreClusterParamSet.fetch(as_dict=True))
all_preparams

,paramset_idx,precluster_method,paramset_desc,param_set_hash,params
0,0,catgt,Catgt Paramset for PassiveTask,4271fe75-1732-5821-0070-45b4c286e867,"{'apfilter': ['biquad', 2, 300, 0], 'gfix': [0..."
1,1,Nothing,Do Nothing method,3d1047b9-deec-bc7f-d5a5-4e30c1665f07,"{'apfilter': ['biquad', 2, 301, 0], 'gfix': [0..."
2,2,dredge,general-user_2026-07-09_dredge default stuff,6ace6f8a-f036-ebf6-1146-d5596e2f6166,"{'preset': 'dredge', 'motion_kwargs': {}, 'n_j..."


### Modify some params from an existing paramset (or load a json or .npy if you have one)

In [4]:
new_param = all_preparams.loc[2, 'params']
new_param

{'preset': 'dredge', 'motion_kwargs': {}, 'n_jobs': 1}

In [5]:
new_param['n_jobs'] = 2

### Set param idx

In [7]:
paramset_idx = all_preparams['paramset_idx'].max() + 1

### Define precluster_method

In [8]:
precluster_method = 'dredge'

### Insert new PreCluster paramset

In [9]:
ep.ephys_element.PreClusterParamSet.insert_new_params(precluster_method,paramset_idx,"paramset decription",new_param)

### Create new list of preprocessing steps

In [6]:
ephys_element_virtual = dj.create_virtual_module("u19_pipeline_ephys_element","u19_pipeline_ephys_element")

In [7]:
allPreClustersLists = pd.DataFrame(ephys_element_virtual.PreClusterParamSteps.fetch(as_dict=True))
allPreClustersLists

,precluster_param_steps_id,precluster_param_steps_name,precluster_param_steps_desc
0,0,Only CatGt,general-user_2022-06-01_Catgt default params
1,1,DoNothing Preprocessing,general-user-2025-11-03_Nothing Preprocessing
2,2,dredge,general-user_2026-07-09__dredge
3,3,dredge,general-user_2026-07-09__dredge


### Insert new "list" of preprocessing

In [17]:
new_list = dict()
new_list['precluster_param_steps_id'] = int(allPreClustersLists['precluster_param_steps_id'].max()+1)
new_list['precluster_param_steps_name'] = 'new list of preprocessing'
new_list['precluster_param_steps_desc'] = 'for testing purposes'
ephys_element_virtual.PreClusterParamSteps.insert1(new_list)

### Insert new "steps" for the newly added preprocessing list

In [20]:
step1 = dict()
step1['precluster_param_steps_id'] = new_list['precluster_param_steps_id'] ## Reference to newly added list idx
step1['step_number'] = 1 ## This preprocessing is done in 1st place
step1['paramset_idx'] = 2 ## This is the reference of PreClusterParamSet record (in this case dredge # 2 params)

step2 = dict()
step2['precluster_param_steps_id'] = new_list['precluster_param_steps_id'] ## Reference to newly added list idx
step2['step_number'] = 2 ## This preprocessing is done in 2nd place
step2['paramset_idx'] = 0 ## This is the reference of PreClusterParamSet record (in this case catgt # 0 params)

all_steps = [step1, step2]
all_steps

[{'precluster_param_steps_id': 4, 'step_number': 1, 'paramset_idx': 2},
 {'precluster_param_steps_id': 4, 'step_number': 2, 'paramset_idx': 0}]

In [21]:
ephys_element_virtual.PreClusterParamSteps.Step.insert(all_steps)

In [8]:
pd.DataFrame(ephys_element_virtual.PreClusterParamSteps.Step.fetch())

,precluster_param_steps_id,step_number,paramset_idx
0,0,1,0
1,1,1,1
2,2,1,2
3,3,1,2
